In [1]:
import os
import geopandas as gpd
import pandas as pd
import rasterio 
from dotenv import load_dotenv
import geopandas as gpd
import numpy as np


load_dotenv()

DATASET_ROOT = os.environ["DATASET_ROOT"]

In [2]:
for i in range(10):
    full_path = os.path.join(DATASET_ROOT, "Xtif/X_{}.tif".format(i))
    try:
        src = rasterio.open(full_path)
        print(set(list(src.descriptions)))
    except Exception as e:
        print(e)

/scratch/nathan/data/hedgementation_1.2/Xtif/X_0.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_1.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_2.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_3.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_4.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_5.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_6.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_7.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_8.tif: No such file or directory
/scratch/nathan/data/hedgementation_1.2/Xtif/X_9.tif: No such file or directory


In [3]:
import re
from datetime import datetime, timedelta

def fix_band_name(band_name):
    match = re.match(r'(\d{4})-(\d{2})-(\d+)(_.*)', band_name)
    if not match:
        return band_name  
    
    week_year, month, day_of_year, suffix = match.groups()
    week_year = int(week_year)
    month = int(month)
    day_of_year = int(day_of_year)
    
    base_date = datetime(week_year, 1, 1)
    
    actual_date = base_date + timedelta(days=day_of_year - 1)
    
    correct_date = actual_date.strftime('%Y-%m-%d')
    
    return correct_date.replace("-","")


In [4]:
def band_names_to_dates(band_names):
    band_names = list(band_names)
    return sorted([
        fix_band_name(bn) for bn in band_names
    ])[::10]

def get_dates_S2(ind, X_path="Xtif/X_{}.tif"):
    full_path = os.path.join(DATASET_ROOT, X_path.format(ind))

    src = rasterio.open(full_path)

    dates_s2 = band_names_to_dates(src.descriptions)  

    if len(dates_s2) != len(src.descriptions) // 10:
        raise Exception(f"Error for {ind}: dates is {len(dates_s2)}, descriptions / 10 is {len(src.descriptions) // 10}") 

    else:
        return {str(i):int(fix_band_name(d)) for i,d in enumerate(dates_s2)}





In [5]:
rows = []
ex = []

# for i in range(2500):
#     rows.append({
#         "ID_PATCH": i,
#         "dates-S2": get_dates_S2(i)
#     })


In [6]:
minval = float("inf")
maxval = float("-inf")
for r in rows:
    minval = min(minval, len([k for k in r["dates-S2"]]))
    maxval = max(maxval, len([k for k in r["dates-S2"]]))
minval, maxval

(inf, -inf)

In [7]:
#gdf = gpd.GeoDataFrame(rows)


In [8]:
gdf = gpd.read_file(f"{DATASET_ROOT}/metadata.geojson")


In [9]:

def get_hedgerow_density(id):
    arr = np.load(f"{DATASET_ROOT}/y/y_{id}.npy")
    return np.sum(arr)

metadata = gpd.read_file(f"{DATASET_ROOT}/metadata.geojson")

hedge_areas = metadata["ID_PATCH"].apply(lambda x: get_hedgerow_density(x))
metadata["hedge_pixel_count"] = hedge_areas

metadata.to_file(f"{DATASET_ROOT}/metadata.geojson")

In [10]:
metadata["hedge_pixel_count"].sum()

np.float64(1139822.2769467584)

In [11]:
metadata["thz_class"].value_counts()

thz_class
TRC7: Temperate, cool                 2310
TRC5: Subtropics, cool                 439
TRC4: Subtropics, moderately cool      233
TRC8: Boreal / Cold, no permafrost      11
TRC6: Temperate, moderately cool         2
Name: count, dtype: int64

In [14]:
len(metadata[metadata["fold"] == 4])

599

In [12]:
metadata.columns

Index(['hedgerows', 'aez_class', 'thz_class', 'mst_class', 'fold',
       'plot_color', 'quadrant', 'sixteenth', 'tile', 'tile_group', 'split',
       'dates-S2', 'ID_PATCH', 'hedge_pixel_count', 'geometry'],
      dtype='object')